In [1]:
import sys
print(sys.executable)


d:\AI ML\04. project\smart_logistics_management\venv\Scripts\python.exe


In [19]:
import sys
from pathlib import Path
import os
import pandas as pd

Path.cwd()

WindowsPath('d:/AI ML/04. project/smart_logistics_management/notebooks')

In [11]:
project_root = Path.cwd().parent
project_root

WindowsPath('d:/AI ML/04. project/smart_logistics_management')

In [12]:
sys.path.append(str(project_root))

In [13]:
Path.cwd()

WindowsPath('d:/AI ML/04. project/smart_logistics_management/notebooks')

In [14]:
from src.etl.pipeline import base_path

In [30]:
# Read shipments.json (handle normal or line-delimited JSON)
try:
    df = pd.read_json(os.path.join(base_path, 'shipments.json'))
except ValueError:
    df = pd.read_json(os.path.join(base_path, 'shipments.json'), lines=True)

# Find rows where shipment_id is duplicated
dups = df[df.duplicated(subset=["shipment_id"], keep=False)]

print("Duplicate shipment_id rows:")
print(dups.sort_values("shipment_id"))
print("\nCount of duplicate shipment_id rows:", len(dups))
dup_ids = dups["shipment_id"].unique()

Duplicate shipment_id rows:
     shipment_id  order_date           origin     destination  weight  \
4553    c2c0ddd0  2026-01-31      Brookeburgh      West Jared    5.89   
8106    c2c0ddd0  2025-04-09  South Brianside  East Johnshire   33.55   

     courier_id     status delivery_date  
4553   995299ac  Cancelled           NaN  
8106   1cddbd0e  Delivered    2025-04-16  

Count of duplicate shipment_id rows: 2


In [31]:
type(dup_ids)

pandas.arrays.ArrowStringArray

In [41]:
# Read shipment_tracking.csv to see duplicate shipment id
# View duplicate shipment_id in shipment_tracking table

df_t = pd.read_csv(os.path.join(base_path, 'shipment_tracking.csv'))

for shipment_id in dup_ids:
    tracking_rows = df_t[df_t["shipment_id"] == shipment_id]
    # parse timestamp so sorting is correct
    tracking_rows["timestamp"] = pd.to_datetime(tracking_rows["timestamp"], errors = "coerce")
    tracking_rows = tracking_rows.sort_values("timestamp", ascending = False)
    print(f"\nTracking rows for shipment_id: {shipment_id}\n")
    print(tracking_rows.to_string())


Tracking rows for shipment_id: c2c0ddd0

       tracking_id shipment_id            status           timestamp
13703        13704    c2c0ddd0         Cancelled 2026-02-02 00:00:00
13702        13703    c2c0ddd0      Order Placed 2026-01-31 02:00:00
24298        24299    c2c0ddd0         Delivered 2025-04-16 12:00:00
24297        24298    c2c0ddd0  Out for Delivery 2025-04-16 07:00:00
24296        24297    c2c0ddd0        In Transit 2025-04-12 00:00:00
24295        24296    c2c0ddd0         Picked Up 2025-04-11 00:00:00
24294        24295    c2c0ddd0      Order Placed 2025-04-09 03:00:00


In [ ]:
# view duplicate id in costs table
df_c = pd.read_csv(os.path.join(base_path, 'costs.csv'))

for shipment_id in dup_ids:
    cost_rows = df_c[df_c["shipment_id"] == shipment_id]
    print(f"\nCost rows for shipment_id: {shipment_id}\n")
    print(cost_rows.to_string()) 


Cost rows for shipment_id: c2c0ddd0

     shipment_id  fuel_cost  labor_cost  misc_cost
4553    c2c0ddd0      77.96       60.66      30.58
8106    c2c0ddd0      47.88       32.56       8.28
